# DSCMFNet Google Colab Training + Testing

This notebook is designed for **Google Colab** and assumes that:

- your repository is available in Google Drive,
- your processed dataset is stored in Google Drive,
- training outputs such as checkpoints, TensorBoard logs, metrics JSON, and prediction previews should also be written to Google Drive.

The notebook supports:

1. mounting Google Drive,
2. installing dependencies,
3. configuring paths and training arguments,
4. launching training with `train.py`,
5. loading a saved checkpoint for evaluation,
6. exporting prediction visualizations back to Google Drive.

## 1. Mount Google Drive and enter the repo

Update `DRIVE_REPO_PATH` if your repository lives somewhere else inside Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import os


drive.mount('/content/drive')

# Example: /content/drive/MyDrive/early_gastric_cancer_detection
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/early_gastric_cancer_detection')
assert DRIVE_REPO_PATH.exists(), f'Repo path not found: {DRIVE_REPO_PATH}'

os.chdir(DRIVE_REPO_PATH)
print('Working directory:', Path.cwd())

## 2. Install dependencies

If you keep a persistent virtual environment elsewhere, feel free to skip this cell.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'tensorboard', 'matplotlib', 'pandas'], check=True)


## 3. Configure paths and experiment settings

All important inputs/outputs below point into **Google Drive**.

In [ ]:
from pathlib import Path
import json

# ----------------------------
# Google Drive data locations
# ----------------------------
DATA_ROOT = Path('/content/drive/MyDrive/egc_data/processed_data')
OUTPUT_ROOT = Path('/content/drive/MyDrive/egc_runs/dscmfnet_colab_run')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Optional raw-data location if you also want to preprocess in Colab
RAW_ROOT = Path('/content/drive/MyDrive/egc_data/raw_data')

# ----------------------------
# Training configuration
# ----------------------------
TRAIN_CONFIG = {
    'phase': 1,                 # 1 = same-scale, 2 = cross-scale
    'model': 'dscmfnet',        # nbi_only | wli_only | early_fusion | dscmfnet
    'fusion_mode': 'concat',    # concat | cmfim | cmfim_no_sam | cmfim_no_boundary
    'epochs': 50,
    'batch_size': 4,
    'img_size': 352,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'freeze_stages': 2,
    'lambda_bdy': 0.5,
    'alpha_ds': 0.3,
    'lambda_wli': 0.0,
    'kfold': 5,
    'seed': 42,
    'num_workers': 2,
    'amp': True,
    'pretrained': True,
    'print_freq': 5,
}

assert DATA_ROOT.exists(), f'Data root not found: {DATA_ROOT}'
print('DATA_ROOT  =', DATA_ROOT)
print('OUTPUT_ROOT=', OUTPUT_ROOT)
print(json.dumps(TRAIN_CONFIG, indent=2))

## 4. Optional preprocessing

Run this only if your Drive contains raw LabelMe/Olympus data and you still need to build `processed_data/`.

In [ ]:
import subprocess

RUN_PREPROCESS = False

if RUN_PREPROCESS:
    preprocess_cmd = [
        'python', 'preprocess.py',
        '--raw_root', str(RAW_ROOT),
        '--out_root', str(DATA_ROOT),
    ]
    print('Running:', ' '.join(preprocess_cmd))
    subprocess.run(preprocess_cmd, check=True)
else:
    print('Skipping preprocessing because RUN_PREPROCESS=False')


## 5. Launch training

This cell writes **checkpoints**, **TensorBoard logs**, and **results.json** into `OUTPUT_ROOT` on Google Drive.

In [ ]:
import shlex
import subprocess

cmd = [
    'python', 'train.py',
    '--data_root', str(DATA_ROOT),
    '--output', str(OUTPUT_ROOT),
    '--phase', str(TRAIN_CONFIG['phase']),
    '--model', TRAIN_CONFIG['model'],
    '--fusion_mode', TRAIN_CONFIG['fusion_mode'],
    '--epochs', str(TRAIN_CONFIG['epochs']),
    '--batch_size', str(TRAIN_CONFIG['batch_size']),
    '--img_size', str(TRAIN_CONFIG['img_size']),
    '--lr', str(TRAIN_CONFIG['lr']),
    '--weight_decay', str(TRAIN_CONFIG['weight_decay']),
    '--freeze_stages', str(TRAIN_CONFIG['freeze_stages']),
    '--lambda_bdy', str(TRAIN_CONFIG['lambda_bdy']),
    '--alpha_ds', str(TRAIN_CONFIG['alpha_ds']),
    '--lambda_wli', str(TRAIN_CONFIG['lambda_wli']),
    '--kfold', str(TRAIN_CONFIG['kfold']),
    '--seed', str(TRAIN_CONFIG['seed']),
    '--num_workers', str(TRAIN_CONFIG['num_workers']),
    '--print_freq', str(TRAIN_CONFIG['print_freq']),
]

if TRAIN_CONFIG['amp']:
    cmd.append('--amp')
cmd.append('--pretrained' if TRAIN_CONFIG['pretrained'] else '--no-pretrained')

cmd_str = ' '.join(shlex.quote(x) for x in cmd)
print('Running command:\n' + cmd_str)
subprocess.run(cmd, check=True)


## 6. Inspect saved outputs on Google Drive

In [ ]:
from pathlib import Path
import json

print('Top-level output files:')
for p in sorted(OUTPUT_ROOT.glob('*')):
    print('-', p)

results_path = OUTPUT_ROOT / 'results.json'
if results_path.exists():
    with open(results_path, 'r', encoding='utf-8') as fh:
        results = json.load(fh)
    print('\nSummary from results.json:')
    print(json.dumps(results.get('summary', {}), indent=2))
else:
    print('results.json not found yet.')


## 7. Load a checkpoint and run evaluation in Colab

This section reuses the repository's Python modules directly so you can test a saved model without starting a new training run.

In [ ]:
import json
from pathlib import Path
import random

import numpy as np
import torch
from torch.utils.data import DataLoader

from train import build_model, validate
from data.dataloader import EGCPhase1Dataset, build_kfold_loaders as build_phase1_loaders, find_pairs
from data.dataloader_crossscale import EGCPhase2Dataset, build_kfold_loaders as build_phase2_loaders, find_crossscale_pairs
from models.losses import DSCMFNetLoss

# Select which fold/checkpoint to evaluate.
EVAL_FOLD = 0
CHECKPOINT_PATH = OUTPUT_ROOT / f'fold_{EVAL_FOLD}' / 'best.pth'
assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'

ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
ckpt_args = ckpt.get('args', {})
print(json.dumps(ckpt_args, indent=2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(
    ckpt_args.get('model', 'dscmfnet'),
    pretrained=False,
    fusion_mode=ckpt_args.get('fusion_mode', 'concat'),
).to(device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

criterion = DSCMFNetLoss(
    lambda_bdy=ckpt_args.get('lambda_bdy', 0.5),
    alpha_ds=ckpt_args.get('alpha_ds', 0.3),
)

phase = int(ckpt_args.get('phase', 1))
kfold = int(ckpt_args.get('kfold', 5))
seed = int(ckpt_args.get('seed', 42))
img_size = int(ckpt_args.get('img_size', 352))
batch_size = int(ckpt_args.get('batch_size', 4))
num_workers = int(ckpt_args.get('num_workers', 2))
lambda_wli = float(ckpt_args.get('lambda_wli', 0.0))
model_type = ckpt_args.get('model', 'dscmfnet')
amp = bool(ckpt_args.get('amp', False)) and device.type == 'cuda'

def build_holdout_loader_phase1(data_root, batch_size, img_size, seed, num_workers):
    pairs = find_pairs(Path(data_root))
    n_val = max(1, int(len(pairs) * 0.2))
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(pairs)).tolist()
    val_pairs = [pairs[i] for i in perm[-n_val:]]
    val_ds = EGCPhase1Dataset(val_pairs, img_size=img_size, augment=False)
    return DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

def build_holdout_loader_phase2(data_root, batch_size, img_size, seed, num_workers):
    pairs = find_crossscale_pairs(Path(data_root))
    n_val = max(1, int(len(pairs) * 0.2))
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(pairs)).tolist()
    val_pairs = [pairs[i] for i in perm[-n_val:]]
    val_ds = EGCPhase2Dataset(val_pairs, img_size=img_size, augment=False)
    return DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

if phase == 1 and kfold > 1:
    _, val_loader = build_phase1_loaders(
        data_root=DATA_ROOT,
        fold_idx=EVAL_FOLD,
        n_folds=kfold,
        batch_size=batch_size,
        img_size=img_size,
        seed=seed,
        num_workers=num_workers,
    )
elif phase == 2 and kfold > 1:
    _, val_loader = build_phase2_loaders(
        data_root=DATA_ROOT,
        fold_idx=EVAL_FOLD,
        n_folds=kfold,
        batch_size=batch_size,
        img_size=img_size,
        seed=seed,
        num_workers=num_workers,
    )
elif phase == 1:
    val_loader = build_holdout_loader_phase1(DATA_ROOT, batch_size, img_size, seed, num_workers)
else:
    val_loader = build_holdout_loader_phase2(DATA_ROOT, batch_size, img_size, seed, num_workers)

val_metrics = validate(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=device,
    model_type=model_type,
    amp=amp,
    lambda_wli=lambda_wli,
)

print('Validation metrics for checkpoint:', CHECKPOINT_PATH)
print(json.dumps(val_metrics, indent=2))


## 8. Save qualitative prediction previews to Google Drive

This cell exports side-by-side visualizations for a few validation samples.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

PRED_DIR = OUTPUT_ROOT / 'prediction_previews'
PRED_DIR.mkdir(parents=True, exist_ok=True)

model.eval()
num_saved = 0
max_images = 8

with torch.no_grad():
    for batch in val_loader:
        wli = batch['wli'].to(device)
        nbi = batch['nbi'].to(device)
        mask = (batch['nbi_mask'] if 'nbi_mask' in batch else batch['mask']).cpu().numpy()
        case_ids = batch['case_id']

        if model_type == 'nbi_only':
            seg, _, _ = model(nbi)
        elif model_type == 'wli_only':
            seg, _, _ = model(wli)
        else:
            seg, _, _ = model(wli, nbi)
        pred = torch.sigmoid(seg).cpu().numpy()
        wli_np = wli.cpu().numpy()
        nbi_np = nbi.cpu().numpy()

        for i, case_id in enumerate(case_ids):
            def denorm(x):
                mean = np.array([0.485, 0.456, 0.406])[:, None, None]
                std = np.array([0.229, 0.224, 0.225])[:, None, None]
                x = (x * std + mean).clip(0, 1)
                return np.transpose(x, (1, 2, 0))

            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            axes[0].imshow(denorm(wli_np[i]))
            axes[0].set_title(f'{case_id} | WLI')
            axes[1].imshow(denorm(nbi_np[i]))
            axes[1].set_title(f'{case_id} | NBI')
            axes[2].imshow(mask[i, 0], cmap='gray')
            axes[2].set_title('Ground Truth')
            axes[3].imshow(pred[i, 0], cmap='magma', vmin=0, vmax=1)
            axes[3].set_title('Prediction')
            for ax in axes:
                ax.axis('off')
            fig.tight_layout()

            save_path = PRED_DIR / f'{case_id}_preview.png'
            fig.savefig(save_path, dpi=150, bbox_inches='tight')
            plt.close(fig)
            print('Saved', save_path)

            num_saved += 1
            if num_saved >= max_images:
                break
        if num_saved >= max_images:
            break

print(f'Saved {num_saved} preview image(s) to {PRED_DIR}')

## 9. Optional TensorBoard in Colab

TensorBoard logs are written to Google Drive under each fold directory.

In [ ]:
# Uncomment to launch TensorBoard in Colab.
# %load_ext tensorboard
# %tensorboard --logdir {OUTPUT_ROOT}
print('TensorBoard logs are stored under:', OUTPUT_ROOT)